Lab 4 — Two-Body Collisions
Reference Solution

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

UT_ORANGE = "#ff8200"
print("NumPy", np.__version__)

Part A — 1D Collisions with Restitution

A1 — The collision function

The center-of-mass form handles every value of *e* with one expression, and makes
momentum conservation structurally obvious: **v_cm never changes.**

$$v_{cm} = \frac{m_1u_1+m_2u_2}{m_1+m_2}, \qquad
v_1 = v_{cm} + e\frac{m_2}{M}(u_2-u_1), \qquad
v_2 = v_{cm} + e\frac{m_1}{M}(u_1-u_2)$$

In [ ]:
def collide_1d(m1, m2, u1, u2, e=1.0):
    """Post-collision velocities for a 1D impact with restitution e.

    e = 1 -> perfectly elastic (KE conserved)
    e = 0 -> perfectly inelastic (bodies move together at v_cm)
    """
    v_cm = (m1*u1 + m2*u2) / (m1 + m2)      # unchanged by the collision
    v1 = v_cm + e * (m2/(m1+m2)) * (u2 - u1)
    v2 = v_cm + e * (m1/(m1+m2)) * (u1 - u2)
    return v1, v2


def p_total(m1, m2, v1, v2):
    return m1*v1 + m2*v2


def ke_total(m1, m2, v1, v2):
    return 0.5*m1*v1**2 + 0.5*m2*v2**2

A2 — The three limiting cases (e = 1)

In [ ]:
cases = [
    ("equal masses",   1.0,   1.0, 3.0, 0.0),
    ("light -> heavy", 0.1, 100.0, 5.0, 0.0),
    ("heavy -> light", 100.0, 0.1, 5.0, 0.0),
]

print(f"{'case':<16s}{'v1':>10s}{'v2':>10s}{'dp':>14s}{'dKE':>14s}")
print("-"*64)
for label, m1, m2, u1, u2 in cases:
    v1, v2 = collide_1d(m1, m2, u1, u2, e=1.0)
    dp  = p_total(m1, m2, v1, v2)  - p_total(m1, m2, u1, u2)
    dke = ke_total(m1, m2, v1, v2) - ke_total(m1, m2, u1, u2)
    print(f"{label:<16s}{v1:10.5f}{v2:10.5f}{dp:14.2e}{dke:14.2e}")


Reading the rows as physics.  Equal masses *exchange* velocities exactly — the
first ball stops dead. The light ball rebounds at 4.99 m/s, very nearly reversing,
having barely disturbed the 100 kg target (0.00999 m/s). The heavy ball slows
imperceptibly (5 → 4.99) while launching the light one at 9.99 m/s — essentially
*twice* the incoming speed. That factor of 2 is the limit $v_2 \to 2u_1$ as
$m_2/m_1 \to 0$, and it is the same physics as a golf club driving a ball.

The residuals in the last two columns are $10^{-13}$ or smaller, which is
floating-point round-off on quantities of order 1–1000, not physics.

A3 — The conservation table. 2 kg block at 4 m/s strikes a stationary 3 kg block.

In [ ]:
m1, m2 = 2.0, 3.0
u1, u2 = 4.0, 0.0

p0  = p_total(m1, m2, u1, u2)
ke0 = ke_total(m1, m2, u1, u2)
v_cm = (m1*u1 + m2*u2) / (m1 + m2)

print(f"before:  p = {p0:.4f} kg m/s   KE = {ke0:.4f} J")
print(f"v_cm  = {v_cm:.4f} m/s  (never changes)\n")

print(f"{'e':>6s}{'v1':>9s}{'v2':>9s}{'p':>10s}{'KE':>10s}{'KE lost':>10s}{'% lost':>9s}")
print("-"*63)
for e in [1.0, 0.75, 0.5, 0.25, 0.0]:
    v1, v2 = collide_1d(m1, m2, u1, u2, e)
    p  = p_total(m1, m2, v1, v2)
    ke = ke_total(m1, m2, v1, v2)
    print(f"{e:6.2f}{v1:9.4f}{v2:9.4f}{p:10.4f}{ke:10.4f}"
          f"{ke0-ke:10.4f}{100*(1-ke/ke0):8.1f}%")

The momentum column is a constant. The KE column is not. That single table is
the whole of Part A.

A4 — Where did the missing kinetic energy go?

The "lost" kinetic energy is not destroyed. It is converted into internal forms the
model does not track: permanent deformation of the blocks, elastic waves that decay
into thermal motion (heat), and the small fraction radiated as sound. Momentum has
nowhere comparable to hide — it is a vector tied to the system as a whole, and the
internal forces between the blocks come in Newton's-third-law pairs that cancel
exactly. Kinetic energy is a scalar that can convert into microscopic degrees of
freedom, and that asymmetry is precisely why momentum is the more reliable
conservation law for collisions.

Two structural observations worth making explicitly:

The $e = 0$ row is $v_{cm}$. Both bodies emerge at exactly 1.6000 m/s. Perfectly
inelastic means "everything ends up moving with the center of mass," and the
remaining 6.4 J is $\tfrac12 M v_{cm}^2$ — the kinetic energy locked in the
center-of-mass motion. You cannot remove it without violating momentum
conservation, which is why 60% is the maximum possible loss here, not an
arbitrary number.

$v_1$ changes sign between $e = 0.75$ and $e = 0.5$. There is an exact value of
$e$ at which the incoming block stops dead: setting $v_1 = 0$ gives
$e = v_{cm}M/(m_2(u_1-u_2)) = 2/3$.

In [ ]:
v1_stop, v2_stop = collide_1d(m1, m2, u1, u2, e=2/3)
print(f"At e = 2/3 exactly:  v1 = {v1_stop:.3e} m/s,  v2 = {v2_stop:.4f} m/s")
print(f"Check: is v1 zero to machine precision? {abs(v1_stop) < 1e-14}")
print(f"\nKE locked in cm motion = {0.5*(m1+m2)*v_cm**2:.4f} J   (= the e=0 row)")

Part B — 2D Collisions in a Box

B1 — Body structure and the line-of-centers decomposition

Each body is a dictionary with `m`, `r` (position 2-vector), `v` (velocity
2-vector), and `radius`. Bodies live in a **list**. This is deliberately the same
structure that becomes `Body` and `World` in Week 7.

The physics: only the velocity component **along the line of centers** is changed by
the collision. The tangential component passes through untouched (no friction, no
spin). With $\hat n$ the unit vector from body 1 to body 2, the impulse magnitude is

$$J = \frac{(1+e)\,(\vec v_{rel}\cdot\hat n)}{1/m_1 + 1/m_2}$$

and it is applied as $\vec v_1 \to \vec v_1 - (J/m_1)\hat n$,
$\vec v_2 \to \vec v_2 + (J/m_2)\hat n$. Because the two impulses are equal and
opposite, momentum conservation is **structural** — it cannot be broken by a sign
error in $J$, only by a wrong $\hat n$.

In [ ]:
def resolve_2d(b1, b2, e=1.0):
    """Apply a collision impulse along the line of centers. Returns True if resolved."""
    n = b2["r"] - b1["r"]
    dist = np.linalg.norm(n)
    if dist == 0:
        return False
    n = n / dist

    v_rel = b1["v"] - b2["v"]
    v_n = np.dot(v_rel, n)          # approach speed along the line of centers

    if v_n <= 0:                    # already separating -- do NOT resolve again
        return False

    m1, m2 = b1["m"], b2["m"]
    J = (1 + e) * v_n / (1/m1 + 1/m2)

    b1["v"] = b1["v"] - (J/m1) * n
    b2["v"] = b2["v"] + (J/m2) * n
    return True


def overlapping(b1, b2):
    return np.linalg.norm(b2["r"] - b1["r"]) < (b1["radius"] + b2["radius"])


def separate(b1, b2):
    """Positional correction, split by INVERSE mass so no momentum is added."""
    d = b2["r"] - b1["r"]
    dist = np.linalg.norm(d)
    if dist == 0:
        return
    n = d / dist
    overlap = (b1["radius"] + b2["radius"]) - dist
    if overlap <= 0:
        return
    im1, im2 = 1/b1["m"], 1/b2["m"]
    b1["r"] = b1["r"] - n * overlap * (im1/(im1+im2))
    b2["r"] = b2["r"] + n * overlap * (im2/(im1+im2))


def p_tot(bodies):
    return sum(b["m"] * b["v"] for b in bodies)


def ke_tot(bodies):
    return sum(0.5 * b["m"] * np.dot(b["v"], b["v"]) for b in bodies)

> The `v_n <= 0` guard is the single most important line in this function.
> Without it, a pair that is overlapping but already separating gets "resolved"
> again on every timestep, which pumps energy into the system. The classic symptom
> is a collision count that goes *up* as you lower *e*, which is physically
> backwards.

B2 — The 90-degree test

For equal masses, one initially at rest, and $e = 1$, the outgoing velocities are
perpendicular at *every* impact parameter. The proof is one line: momentum gives
$\vec u = \vec v_1 + \vec v_2$ and energy gives $u^2 = v_1^2 + v_2^2$; squaring the
first and subtracting leaves $2\,\vec v_1\cdot\vec v_2 = 0$.

This is the best unit test in the lab because it is *geometry-independent — it
catches normal-vector bugs that a head-on test would sail straight past.

In [ ]:
print("Equal masses, m2 at rest, e = 1 -- testing several impact parameters\n")
print(f"{'b':>6s}{'v1 . v2':>14s}{'angle (deg)':>14s}")
print("-"*34)

for b in [0.2, 0.4, 0.6, 0.8]:
    dx = np.sqrt(1.0 - b**2)                 # place them exactly at contact
    b1 = {"m":1.0, "r":np.array([-dx, 0.0]), "v":np.array([4.0, 0.0]), "radius":0.5}
    b2 = {"m":1.0, "r":np.array([0.0,  b ]), "v":np.array([0.0, 0.0]), "radius":0.5}

    resolve_2d(b1, b2, e=1.0)

    dot = np.dot(b1["v"], b2["v"])
    n1, n2 = np.linalg.norm(b1["v"]), np.linalg.norm(b2["v"])
    angle = np.degrees(np.arccos(np.clip(dot/(n1*n2), -1, 1)))
    print(f"{b:6.1f}{dot:14.3e}{angle:14.3f}")

The dot product is zero to machine precision regardless of how the balls are aimed.
Now confirm the test *fails correctly* when the collision is inelastic — if it
passed at $e < 1$, it would not be testing anything.

In [ ]:
print("Same geometry (b = 0.6), varying e -- the 90-deg result should BREAK\n")
print(f"{'e':>6s}{'v1 . v2':>12s}{'angle (deg)':>14s}")
print("-"*32)
for e in [1.0, 0.75, 0.5, 0.25]:
    b1 = {"m":1.0, "r":np.array([-0.8, 0.0]), "v":np.array([4.0, 0.0]), "radius":0.5}
    b2 = {"m":1.0, "r":np.array([0.0,  0.6]), "v":np.array([0.0, 0.0]), "radius":0.5}
    resolve_2d(b1, b2, e)
    dot = np.dot(b1["v"], b2["v"])
    n1, n2 = np.linalg.norm(b1["v"]), np.linalg.norm(b2["v"])
    angle = np.degrees(np.arccos(np.clip(dot/(n1*n2), -1, 1)))
    print(f"{e:6.2f}{dot:12.4f}{angle:14.2f}")

The separation angle closes as energy is removed. At $e = 0$ the two bodies move
together and the angle would be exactly 0. This is the correct failure mode, so the
test is doing real work.

B3 — The simulation loop  
Note the deliberate ordering: move → walls → pairs. 

In [ ]:
def make_bodies():
    """Four balls in a 10 x 10 box. Deterministic -- this is your future reset()."""
    specs = [
        (1.0, [2.0, 2.0], [ 3.0,  2.5], 0.40),
        (1.0, [8.0, 3.0], [-2.5,  1.5], 0.40),
        (2.0, [5.0, 7.0], [ 1.5, -3.0], 0.50),
        (1.5, [3.0, 8.0], [-2.0, -2.0], 0.45),
    ]
    return [{"m":m, "r":np.array(r, dtype=float),
             "v":np.array(v, dtype=float), "radius":rad}
            for m, r, v, rad in specs]


def run(bodies, dt, t_max, e=1.0, L=10.0, walls=True, record_every=None):
    """Advance the system. Returns (n_pair_collisions, n_wall_hits, history)."""
    n_steps = int(t_max / dt)
    n_coll = n_wall = 0
    history = []

    for step in range(n_steps):
        # 1. move
        for b in bodies:
            b["r"] = b["r"] + b["v"] * dt

        # 2. walls (elastic; an EXTERNAL force -- breaks momentum, not energy)
        if walls:
            for b in bodies:
                for k in (0, 1):
                    if b["r"][k] - b["radius"] < 0:
                        b["r"][k] = b["radius"]
                        b["v"][k] = -b["v"][k]
                        n_wall += 1
                    elif b["r"][k] + b["radius"] > L:
                        b["r"][k] = L - b["radius"]
                        b["v"][k] = -b["v"][k]
                        n_wall += 1

        # 3. pairs: detect -> resolve -> correct position
        for i in range(len(bodies)):
            for j in range(i+1, len(bodies)):
                if overlapping(bodies[i], bodies[j]):
                    if resolve_2d(bodies[i], bodies[j], e):
                        n_coll += 1
                    separate(bodies[i], bodies[j])

        if record_every and step % record_every == 0:
            history.append((step*dt, ke_tot(bodies), p_tot(bodies).copy()))

    return n_coll, n_wall, history

B4 — Conservation checks: 
The isolated system: walls OFF

Momentum is conserved only when the system is isolated. A wall is an external
force, so this test must be run with `walls=False` or it is meaningless.

In [ ]:
print("WALLS OFF -- isolated system, momentum MUST be conserved\n")
print(f"{'e':>5s}{'collisions':>12s}{'|dp|/|p0|':>14s}{'KE (% of KE0)':>16s}")
print("-"*47)
for e in [1.0, 0.9, 0.5, 0.0]:
    bodies = make_bodies()
    p0 = p_tot(bodies).copy()
    ke0 = ke_tot(bodies)
    n_c, n_w, _ = run(bodies, dt=0.002, t_max=20.0, e=e, walls=False)
    dp = np.linalg.norm(p_tot(bodies) - p0) / np.linalg.norm(p0)
    print(f"{e:5.1f}{n_c:12d}{dp:14.2e}{100*ke_tot(bodies)/ke0:15.3f}%")

Momentum is conserved to exactly zero at every restitution, which is the
expected result: the impulses are applied as an equal-and-opposite pair, and the
positional correction is split by inverse mass, so neither operation can change
$\sum m\vec v$. Kinetic energy falls monotonically with $e$, and never rises.

The box: walls ON

In [ ]:
print("WALLS ON -- momentum is NOT expected to be conserved (walls are external)\n")
print(f"{'e':>5s}{'pair coll.':>12s}{'wall hits':>11s}{'KE (% of KE0)':>16s}")
print("-"*44)
for e in [1.0, 0.9, 0.5, 0.0]:
    bodies = make_bodies()
    ke0 = ke_tot(bodies)
    n_c, n_w, _ = run(bodies, dt=0.002, t_max=20.0, e=e, walls=True)
    print(f"{e:5.1f}{n_c:12d}{n_w:11d}{100*ke_tot(bodies)/ke0:15.3f}%")



Three things to read here, one of which is a trap:

At $e = 1$, KE is exactly 100.000% even with 38 wall hits. An elastic wall
reverses one velocity component and therefore does *no work* — it changes momentum
without changing energy. That is the cleanest possible illustration of why momentum
and energy are independent diagnostics.

The KE column is not monotonic in $e$ — $e=0$ retains more energy (32%) than
$e=0.5$ (9.9%). This is not a bug. A hard-sphere gas is chaotic: at $e=0$ the
balls stick together into slow-moving clumps that stop colliding (7 pair collisions
instead of 9), so dissipation shuts itself off. Different $e$ values produce
genuinely different trajectories, not perturbations of one trajectory. 

The collision counts are small because four balls in a 10 x 10 box is a dilute
system. Part C uses 25 balls to get statistics.

Timestep sensitivity

In [ ]:
print("Timestep study, e = 1, walls on, t_max = 20 s\n")
print(f"{'dt':>9s}{'pair coll.':>12s}{'KE (% of KE0)':>18s}")
print("-"*39)
for dt in [0.01, 0.005, 0.002, 0.001, 0.0005]:
    bodies = make_bodies()
    ke0 = ke_tot(bodies)
    n_c, n_w, _ = run(bodies, dt=dt, t_max=20.0, e=1.0)
    print(f"{dt:9.4f}{n_c:12d}{100*ke_tot(bodies)/ke0:17.6f}%")

This looks like a contradiction and is not. Kinetic energy is exact at every
timestep, while the collision count refuses to converge.

Both are correct, for different reasons. `resolve_2d` is algebraic — it solves
the impulse equations exactly, so it introduces no truncation error whatsoever;
energy conservation at $e=1$ is guaranteed by the formula, not by the integrator.
Meanwhile the trajectories are chaotic: two nearly-identical initial states
diverge exponentially, so a sub-millisecond difference in when a contact is detected
compounds into a completely different sequence of collisions within a few seconds.

The lesson generalizes well beyond this lab: invariants converge even when
trajectories do not. That is exactly why conservation laws — not positions — are
the right diagnostic for a many-body engine.

Energy history plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for e, color in zip([1.0, 0.9, 0.5], [UT_ORANGE, "#006c93", "#7a3fa0"]):
    bodies = make_bodies()
    ke0 = ke_tot(bodies)
    _, _, hist = run(bodies, dt=0.002, t_max=20.0, e=e,
                     walls=True, record_every=25)
    t  = np.array([h[0] for h in hist])
    ke = np.array([h[1] for h in hist]) / ke0
    axes[0].plot(t, ke, color=color, lw=1.6, label=f"e = {e}")

axes[0].set_xlabel("time (s)"); axes[0].set_ylabel("KE / KE$_0$")
axes[0].set_title("Box with walls: energy vs. time")
axes[0].legend(); axes[0].set_ylim(0, 1.05)

for e, color in zip([1.0, 0.9, 0.5], [UT_ORANGE, "#006c93", "#7a3fa0"]):
    bodies = make_bodies()
    p0 = np.linalg.norm(p_tot(bodies))
    _, _, hist = run(bodies, dt=0.002, t_max=20.0, e=e,
                     walls=False, record_every=25)
    t  = np.array([h[0] for h in hist])
    dp = np.array([np.linalg.norm(h[2]) for h in hist]) / p0
    axes[1].plot(t, dp, color=color, lw=1.6, label=f"e = {e}")

axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("|p| / |p$_0$|")
axes[1].set_title("Walls OFF: momentum is flat at 1.0")
axes[1].set_ylim(0.99, 1.01); axes[1].legend()

plt.tight_layout(); plt.show()

The left panel shows energy stepping down at each inelastic collision — flat
between collisions, because nothing dissipates in free flight. The right panel is a
horizontal line at exactly 1.0 for all three restitutions: momentum does not care
how inelastic the collisions are.

Part C — Independent Investigation 

> Your Part C will not look like mine, and should not. What must match is the
> structure: a question, a prediction, a controlled experiment, a quantitative
> comparison, and an honest limitation. Two investigations are shown here.

C1 

In [ ]:
def make_gas(n=25, L=10.0, seed=7, speed=3.0, radius=0.25):
    """n equal spheres, random directions, identical speeds, no overlaps."""
    rng = np.random.default_rng(seed)
    bodies = []
    while len(bodies) < n:
        r = rng.uniform(radius, L - radius, 2)
        if all(np.linalg.norm(r - b["r"]) > 2.2*radius for b in bodies):
            th = rng.uniform(0, 2*np.pi)
            bodies.append({"m":1.0, "r":r,
                           "v":speed*np.array([np.cos(th), np.sin(th)]),
                           "radius":radius})
    return bodies

print(f"{'e':>6s}{'tau (s)':>10s}{'R2 Haff':>10s}{'R2 exp':>10s}"
      f"{'KE_end/KE0':>13s}{'tau(1-e^2)':>13s}")
print("-"*62)

results = {}
for e in [0.95, 0.90, 0.80]:
    bodies = make_gas()
    ke0 = ke_tot(bodies)
    _, _, hist = run(bodies, dt=0.002, t_max=40.0, e=e,
                     walls=True, record_every=50)
    t  = np.array([h[0] for h in hist])
    ke = np.array([h[1] for h in hist]) / ke0

    # Haff test: 1/sqrt(KE) should be LINEAR in t
    y  = 1/np.sqrt(ke)
    A  = np.polyfit(t, y, 1)
    r2_haff = 1 - np.sum((y - np.polyval(A, t))**2)/np.sum((y - y.mean())**2)

    # competing hypothesis: log(KE) linear in t
    ly = np.log(ke)
    B  = np.polyfit(t, ly, 1)
    r2_exp = 1 - np.sum((ly - np.polyval(B, t))**2)/np.sum((ly - ly.mean())**2)

    tau = 1/A[0]
    results[e] = (t, ke, tau)
    print(f"{e:6.2f}{tau:10.3f}{r2_haff:10.4f}{r2_exp:10.4f}"
          f"{ke[-1]:13.4f}{tau*(1-e**2):13.4f}")


The Haff form wins at $e = 0.95$ and $e = 0.90$ ($R^2 = 0.992$ and
$0.998$ versus $0.974$ and $0.982$ for the exponential). More convincing than either
$R^2$ is the second test: $\tau(1-e^2)$ comes out as **4.23, 3.98, 4.26** across a
factor of four in $\tau$ — constant to about 4%. The predicted scaling holds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = {0.95: UT_ORANGE, 0.90: "#006c93", 0.80: "#7a3fa0"}

for e, (t, ke, tau) in results.items():
    axes[0].plot(t, ke, color=colors[e], lw=1.5, label=f"e = {e}")
    axes[0].plot(t, 1/(1 + t/tau)**2, "--", color=colors[e], lw=1.0, alpha=0.8)
    axes[1].plot(t, 1/np.sqrt(ke), color=colors[e], lw=1.5, label=f"e = {e}")

axes[0].set_xlabel("time (s)"); axes[0].set_ylabel("KE / KE$_0$")
axes[0].set_title("Granular cooling (dashed = Haff prediction)"); axes[0].legend()

axes[1].set_xlabel("time (s)"); axes[1].set_ylabel(r"$1/\sqrt{KE/KE_0}$")
axes[1].set_title("Haff linearization: straight lines confirm the law")
axes[1].legend()

plt.tight_layout(); plt.show()

C2 
Question. A 100 kg body moving at 10 m/s must transfer energy to a 1 kg body.
Hitting it directly is inefficient (huge mass mismatch). If one intermediate mass
$m_2$ is inserted in a collinear chain, what value maximizes the final speed of the
1 kg body?
Method. Scan $m_2$ over a fine grid, using the validated `collide_1d` twice.

In [ ]:
m1, m3 = 100.0, 1.0
u1 = 10.0

def two_stage(m2):
    """100 kg hits m2; m2 then hits the 1 kg body. All elastic, all collinear."""
    _, v2 = collide_1d(m1, m2, u1, 0.0, e=1.0)
    _, v3 = collide_1d(m2, m3, v2, 0.0, e=1.0)
    return v3

masses = np.linspace(0.5, 40.0, 4000)
finals = np.array([two_stage(m) for m in masses])
i_best = np.argmax(finals)

_, v3_direct = collide_1d(m1, m3, u1, 0.0, e=1.0)

print(f"optimal m2 (numerical)   = {masses[i_best]:.4f} kg")
print(f"geometric mean sqrt(m1*m3) = {np.sqrt(m1*m3):.4f} kg")
print(f"relative error            = {abs(masses[i_best]-10)/10*100:.3f} %\n")
print(f"final speed, two-stage    = {finals[i_best]:.4f} m/s")
print(f"final speed, direct hit   = {v3_direct:.4f} m/s")
print(f"speed gain                = {finals[i_best]/v3_direct:.4f} x")
print(f"energy gain               = {(finals[i_best]/v3_direct)**2:.4f} x")


The numerical optimum lands on the geometric mean to 0.02%, which is
grid resolution, not physics. Inserting a single well-chosen intermediate mass
delivers 2.79 times the kinetic energy of a direct hit — and it does so without
adding any energy to the system, purely by staging the momentum transfer.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(masses, finals, color=UT_ORANGE, lw=2, label="two-stage final speed")
ax.axvline(np.sqrt(m1*m3), color="#006c93", ls="--", lw=1.4,
           label=r"$\sqrt{m_1 m_3} = 10$ kg")
ax.axhline(v3_direct, color="#888888", ls=":", lw=1.4,
           label=f"direct hit ({v3_direct:.2f} m/s)")
ax.set_xlabel("intermediate mass $m_2$ (kg)")
ax.set_ylabel("final speed of 1 kg body (m/s)")
ax.set_title("Impedance matching in a collision chain")
ax.legend(); plt.tight_layout(); plt.show()